# 15f. Hybrid GAT + LightGBM, out-of-fold chia XEN KẼ trên NF-ToN-IoT-v3, GPU (task 5.7, GĐ F)

**Nguyên nhân hybrid out-of-fold thất bại (từ log `classes_missing_in_fit` của notebook 15b):** 5 nhóm liên tiếp theo thời gian, mà mỗi loại tấn công chỉ xuất hiện ở một giai đoạn: nhóm cuối của train là nơi duy nhất có DoS_DDoS (BoT) và Botnet_Backdoor, Khac (ToN). GAT huấn luyện thiếu nhóm cuối chưa từng thấy các lớp đó (F1 val riêng của GAT này chỉ 0,57–0,67, bốn GAT còn lại 0,89–0,94), nên xác suất và embedding out-of-fold của nhóm cuối vô nghĩa.

**Sửa:** cắt train thành 5×20 = 100 khối liên tiếp theo thời gian, khối i thuộc nhóm i mod 5; mọi nhóm trải khắp mọi giai đoạn nên mọi GAT thấy đủ lớp. Chạy cả hai dạng với cùng GAT nhóm: `hybrid_gat_lgbm_oofavg_il` (embedding 64 chiều, val = trung bình 5 góc) và `hybrid_gat_lgbm_stack_il` (xác suất lớp). 3 seed, chỉ `split=val`, **không đọc test**.

Notebook này chỉ chạy ToN và chỉ dạng embedding (`hybrid_gat_lgbm_oofavg_il`), **GPU T4**, Phúc đã đồng ý (15/11). Bỏ GAT toàn train chẩn đoán để tiết kiệm giờ GPU. Dạng xác suất (`stack_il`) chạy sau ở notebook CPU từ checkpoint của notebook này. Gắn output của 04, 11. Bật Internet.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

DATASETS = ["nf_ton_iot_v3"]
ALLOW_GPU = True
INTERLEAVE = 20

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None, "datasets": DATASETS, "interleave": INTERLEAVE}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"])
assert ALLOW_GPU or INFO["device"] == "CPU", "Chưa được phép dùng GPU: tắt GPU trong cài đặt notebook"

## 1. pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Out-of-fold xen kẽ: embedding (oofavg_il) rồi xác suất (stack_il, dùng lại GAT nhóm vừa huấn luyện)

In [ ]:
MODELS = f"{WORK}/models_hybrid"
for ds in DATASETS:
    data_dir = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    avg_res, avg_search = f"{WORK}/results_hybrid_oofavg_il_{ds}.csv", f"{WORK}/hybrid_search_oofavg_il_{ds}.json"
    for p in (avg_res, avg_search):
        assert not os.path.exists(p), f"{p} đã có từ trước"
    print("=====", ds, "| dữ liệu:", data_dir)
    with step(f"oofavg_il_{ds}"):
        run([sys.executable, "-I", "scripts/run_hybrid_oofavg.py", data_dir, ds, avg_res, avg_search,
             f"models={MODELS}", f"interleave={INTERLEAVE}", "fulldiag=0"], cwd=REPO)

## 3. Kiểm tra và run_info.json (so sánh với baseline làm ở notebook 15c từ file trong repo)

In [ ]:
import pandas as pd

crit, diag, folds = {}, {}, {}
for ds in DATASETS:
    crit[ds] = {}
    for tag, key in (("oofavg_il", "hybrid_gat_lgbm_oofavg_il"),):
        res = pd.read_csv(f"{WORK}/results_hybrid_{tag}_{ds}.csv")
        cnt = res.groupby(["model", "seed"]).size()
        crit[ds][tag] = {"du_3_seed_1_dong": bool(len(res) == 3 and (cnt == 1).all() and set(res["seed"]) == {42, 43, 44}),
                         "chi_split_val": set(res["split"]) == {"val"},
                         "f1_macro_mean": round(float(res["f1_macro"].mean()), 4),
                         "f1_macro_std": round(float(res["f1_macro"].std(ddof=1)), 4),
                         "fpr_mean": round(float(res["fpr"].mean()), 4), "recall_mean": round(float(res["recall_attack"].mean()), 4),
                         "size_kb_max": float(res["size_kb"].max()), "p95_ms_max": float(res["infer_ms_per_window"].max())}
        s = json.load(open(f"{WORK}/hybrid_search_{tag}_{ds}.json", encoding="utf-8"))[key]["seeds"]
        diag[f"{ds}|{tag}"] = {k: v["diag"] for k, v in s.items()}
        if tag == "oofavg_il":
            folds[ds] = {k: [f["classes_missing_in_fit"] for f in v["folds"]] for k, v in s.items()}
    print(ds, json.dumps(crit[ds], ensure_ascii=False, indent=1))
    print("lớp vắng trong train của từng GAT nhóm:", folds[ds])
    print(json.dumps({k: v for k, v in diag.items() if k.startswith(ds)}, ensure_ascii=False, indent=1))
INFO["criteria"], INFO["diag"], INFO["classes_missing_in_fit"] = crit, diag, folds
INFO["models_hybrid_dirs"] = sorted(os.listdir(MODELS))
save_info()
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())